In [7]:
!pip install fastapi uvicorn[standard] nest_asyncio pyngrok fakeredis mlflow scikit-learn xgboost lightgbm shap structlog prometheus-client pydantic-settings
!pip install pydantic-settings


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.2 MB/s eta 0:00:00


In [8]:
from redis import Redis
import fakeredis


In [14]:
import logging
import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

# Explainability
import shap

# Database and Storage
import sqlite3
from redis import Redis
import mlflow
import mlflow.sklearn

# API Framework
from fastapi import FastAPI, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, validator
import uvicorn

# Monitoring and Logging
from prometheus_client import Counter, Histogram, Gauge
import structlog

# Configuration
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    """Application settings"""
    redis_url: str = "redis://localhost:6379"
    database_url: str = "sqlite:///insurance.db"
    mlflow_tracking_uri: str = "sqlite:///mlflow.db"
    model_registry_name: str = "insurance_risk_model"
    log_level: str = "INFO"

    class Config:
        env_file = ".env"

settings = Settings()

# Configure logging
logging.basicConfig(level=getattr(logging, settings.log_level))
logger = structlog.get_logger()

# Metrics
from prometheus_client import CollectorRegistry, Counter, Histogram, Gauge

# create a fresh registry so Colab re‑runs don’t collide
registry = CollectorRegistry()

risk_assessment_counter = Counter(
    'risk_assessments_total',
    'Total risk assessments',
    registry=registry
)
risk_assessment_duration = Histogram(
    'risk_assessment_duration_seconds',
    'Risk assessment duration',
    registry=registry
)
model_prediction_accuracy = Gauge(
    'model_prediction_accuracy',
    'Model prediction accuracy',
    registry=registry
)


# Data Models
class VehicleType(str, Enum):
    SEDAN = "sedan"
    SUV = "suv"
    TRUCK = "truck"
    MOTORCYCLE = "motorcycle"
    SPORTS_CAR = "sports_car"

class CoverageType(str, Enum):
    LIABILITY = "liability"
    COMPREHENSIVE = "comprehensive"
    COLLISION = "collision"
    FULL_COVERAGE = "full_coverage"

@dataclass
class ApplicantData:
    """Core applicant information"""
    age: int
    gender: str
    marital_status: str
    education_level: str
    occupation: str
    annual_income: float
    credit_score: int
    zip_code: str
    years_licensed: int

    # Driving history
    accidents_last_5_years: int
    violations_last_3_years: int
    dui_history: bool
    license_suspensions: int

    # Vehicle information
    vehicle_make: str
    vehicle_model: str
    vehicle_year: int
    vehicle_type: VehicleType
    vehicle_value: float
    safety_rating: float
    anti_theft_devices: bool

    # Coverage preferences
    coverage_type: CoverageType
    deductible_amount: float
    coverage_limit: float

    # Telematics data
    miles_driven_annually: float
    avg_speed: float
    hard_braking_events: int
    hard_acceleration_events: int
    nighttime_driving_pct: float
    highway_driving_pct: float

    # Additional risk factors
    parking_location: str  # garage, driveway, street
    commute_distance: float
    usage_type: str  # personal, business, rideshare

class ApplicantRequest(BaseModel):
    """API request model"""
    applicant_data: Dict[str, Any]
    explanation_required: bool = True

    @validator('applicant_data')
    def validate_applicant_data(cls, v):
        required_fields = [
            'age', 'credit_score', 'zip_code', 'years_licensed',
            'accidents_last_5_years', 'violations_last_3_years',
            'vehicle_year', 'vehicle_value', 'miles_driven_annually'
        ]

        for field in required_fields:
            if field not in v:
                raise ValueError(f"Missing required field: {field}")

        return v

@dataclass
class RiskAssessment:
    """Risk assessment result"""
    applicant_id: str
    risk_score: float
    risk_category: str
    confidence_score: float
    feature_importance: Dict[str, float]
    explanation: str
    assessment_timestamp: datetime
    model_version: str

class FeatureEngineer:
    """Feature engineering for insurance risk assessment"""

    def __init__(self):
        self.scalers = {}
        self.encoders = {}
        self.fitted = False

    def create_derived_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create derived features from base applicant data"""
        df = df.copy()

        # Age-based features
        df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                                labels=['under_25', '25_34', '35_44', '45_54', '55_64', '65_plus'])

        # Experience features
        df['driving_experience'] = df['years_licensed']
        df['experience_per_age'] = df['years_licensed'] / df['age']

        # Vehicle age and depreciation
        current_year = datetime.now().year
        df['vehicle_age'] = current_year - df['vehicle_year']
        df['vehicle_depreciation_rate'] = (df['vehicle_value'] /
                                         (df['vehicle_year'] - 2000 + 1)).clip(0, 1)

        # Risk composite scores
        df['accident_rate'] = df['accidents_last_5_years'] / 5
        df['violation_rate'] = df['violations_last_3_years'] / 3
        df['total_incidents'] = df['accidents_last_5_years'] + df['violations_last_3_years']

        # Credit score tiers
        df['credit_tier'] = pd.cut(df['credit_score'],
                                 bins=[0, 580, 670, 740, 800, 850],
                                 labels=['poor', 'fair', 'good', 'very_good', 'excellent'])

        # Telematics risk factors
        df['aggressive_driving_score'] = (
            (df['hard_braking_events'] + df['hard_acceleration_events']) /
            (df['miles_driven_annually'] / 1000)
        ).clip(0, 10)

        df['high_risk_driving_pct'] = df['nighttime_driving_pct']
        df['annual_mileage_tier'] = pd.cut(df['miles_driven_annually'],
                                         bins=[0, 7500, 15000, 25000, float('inf')],
                                         labels=['low', 'medium', 'high', 'very_high'])

        # Geographic risk (simplified - would use actual geo data)
        df['zip_risk_score'] = df['zip_code'].astype(str).str[-1:].astype(int) / 10

        # Financial stability indicators
        df['income_to_coverage_ratio'] = df['annual_income'] / df['coverage_limit']
        df['deductible_affordability'] = df['deductible_amount'] / df['annual_income']

        return df

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit feature transformers and transform data"""
        df = self.create_derived_features(df)

        # Separate numeric and categorical features
        numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

        # Remove target variable if present
        if 'risk_score' in numeric_features:
            numeric_features.remove('risk_score')

        # Fit and transform numeric features
        if numeric_features:
            self.scalers['numeric'] = StandardScaler()
            df[numeric_features] = self.scalers['numeric'].fit_transform(df[numeric_features])

        # Fit and transform categorical features
        for col in categorical_features:
            self.encoders[col] = LabelEncoder()
            df[col] = self.encoders[col].fit_transform(df[col].astype(str))

        self.fitted = True
        return df

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform new data using fitted transformers"""
        if not self.fitted:
            raise ValueError("FeatureEngineer must be fitted before transform")

        df = self.create_derived_features(df)

        # Transform numeric features
        numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
        if 'risk_score' in numeric_features:
            numeric_features.remove('risk_score')

        if numeric_features and 'numeric' in self.scalers:
            df[numeric_features] = self.scalers['numeric'].transform(df[numeric_features])

        # Transform categorical features
        categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
        for col in categorical_features:
            if col in self.encoders:
                # Handle unseen categories
                df[col] = df[col].astype(str)
                unseen_mask = ~df[col].isin(self.encoders[col].classes_)
                df.loc[unseen_mask, col] = self.encoders[col].classes_[0]  # Default to first class
                df[col] = self.encoders[col].transform(df[col])

        return df

class ModelTrainer:
    """Model training and evaluation for insurance risk assessment"""

    def __init__(self):
        self.models = {}
        self.best_model = None
        self.feature_engineer = FeatureEngineer()
        self.explainer = None

    def generate_synthetic_data(self, n_samples: int = 10000) -> pd.DataFrame:
        """Generate synthetic insurance data for training"""
        np.random.seed(42)

        data = {


            'age': np.random.randint(18, 80, n_samples),
            'gender': np.random.choice(['M', 'F'], n_samples),
            'marital_status': np.random.choice(['single', 'married', 'divorced'], n_samples),
            'education_level': np.random.choice(['high_school', 'college', 'graduate'], n_samples),
            'occupation': np.random.choice(['professional', 'service', 'manual', 'retired'], n_samples),
            'annual_income': np.random.lognormal(10.5, 0.5, n_samples),
            'credit_score': np.random.randint(300, 850, n_samples),
            'zip_code': np.random.randint(10000, 99999, n_samples),
            'years_licensed': np.random.randint(1, 50, n_samples),
            'accidents_last_5_years': np.random.poisson(0.5, n_samples),
            'violations_last_3_years': np.random.poisson(0.3, n_samples),
            'dui_history': np.random.choice([True, False], n_samples, p=[0.05, 0.95]),
            'license_suspensions': np.random.poisson(0.1, n_samples),
            'vehicle_make': np.random.choice(['Toyota', 'Ford', 'BMW', 'Mercedes'], n_samples),
            'vehicle_model': np.random.choice(['Sedan', 'SUV', 'Truck', 'Coupe'], n_samples),
            'vehicle_year': np.random.randint(2010, 2024, n_samples),
            'vehicle_type': np.random.choice(['sedan', 'suv', 'truck', 'sports_car'], n_samples),
            'vehicle_value': np.random.lognormal(9.5, 0.5, n_samples),
            'safety_rating': np.random.uniform(3.0, 5.0, n_samples),
            'anti_theft_devices': np.random.choice([True, False], n_samples, p=[0.7, 0.3]),
            'coverage_type': np.random.choice(['liability', 'comprehensive', 'full_coverage'], n_samples),
            'deductible_amount': np.random.choice([500, 1000, 2000, 5000], n_samples),
            'coverage_limit': np.random.choice([50000, 100000, 250000, 500000], n_samples),
            'miles_driven_annually': np.random.lognormal(9.5, 0.3, n_samples),
            'avg_speed': np.random.normal(45, 10, n_samples),
            'hard_braking_events': np.random.poisson(2, n_samples),
            'hard_acceleration_events': np.random.poisson(1.5, n_samples),
            'nighttime_driving_pct': np.random.uniform(0.1, 0.4, n_samples),
            'highway_driving_pct': np.random.uniform(0.3, 0.8, n_samples),
            'parking_location': np.random.choice(['garage', 'driveway', 'street'], n_samples),
            'commute_distance': np.random.lognormal(2.5, 0.8, n_samples),
            'usage_type': np.random.choice(['personal', 'business'], n_samples, p=[0.9, 0.1])


        }

        df = pd.DataFrame(data)
        df['vehicle_age'] = datetime.now().year - df['vehicle_year']


        # Generate risk score based on features (simplified risk model)
        risk_factors = (
            (df['age'] < 25) * 0.3 +
            (df['age'] > 65) * 0.2 +
            (df['accidents_last_5_years'] * 0.4) +
            (df['violations_last_3_years'] * 0.3) +
            (df['dui_history'] * 0.5) +
            (df['credit_score'] < 600) * 0.2 +
            (df['miles_driven_annually'] > 20000) * 0.1 +
            (df['vehicle_age'] > 10) * 0.1 +
            (df['hard_braking_events'] * 0.02) +
            (df['nighttime_driving_pct'] * 0.2) +
            np.random.normal(0, 0.1, n_samples)  # Random noise
        )

        # Scale risk score to 0-100 range
        df['risk_score'] = np.clip(risk_factors * 20 + 50, 0, 100)

        return df

    def train_models(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Train multiple models and select the best one"""
        logger.info("Starting model training...")

        # Feature engineering
        df_processed = self.feature_engineer.fit_transform(df)

        # Prepare features and target
        X = df_processed.drop(['risk_score'], axis=1)
        y = df_processed['risk_score']

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Define models to train
        models_config = {
            'linear_regression': LinearRegression(),
            'random_forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'gradient_boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
            'xgboost': xgb.XGBRegressor(n_estimators=100, random_state=42),
            'lightgbm': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
        }

        # Train and evaluate models
        results = {}

        for name, model in models_config.items():
            logger.info(f"Training {name}...")

            # Train model
            model.fit(X_train, y_train)

            # Make predictions
            y_pred = model.predict(X_test)

            # Calculate metrics
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            # Cross-validation
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')

            results[name] = {
                'model': model,
                'mse': mse,
                'mae': mae,
                'r2': r2,
                'cv_mse': -cv_scores.mean(),
                'cv_std': cv_scores.std()
            }

            logger.info(f"{name} - MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

        # Select best model based on cross-validation MSE
        best_model_name = min(results.keys(), key=lambda x: results[x]['cv_mse'])
        self.best_model = results[best_model_name]['model']

        logger.info(f"Best model: {best_model_name}")

        # Create SHAP explainer for best model
        if hasattr(self.best_model, 'predict'):
            sample_data = X_train.sample(min(1000, len(X_train)))
            self.explainer = shap.Explainer(self.best_model, sample_data)

        self.models = results
        return results

    def save_model(self, model_name: str = "insurance_risk_model"):
        """Save the trained model and feature engineer"""
        if self.best_model is None:
            raise ValueError("No model has been trained yet")

        # Save with MLflow
        mlflow.set_tracking_uri(settings.mlflow_tracking_uri)

        with mlflow.start_run():
            # Log model
            mlflow.sklearn.log_model(
                self.best_model,
                model_name,
                registered_model_name=settings.model_registry_name
            )

            # Log feature engineer
            mlflow.log_artifact("feature_engineer.pkl")

            # Log metrics
            for name, results in self.models.items():
                mlflow.log_metric(f"{name}_mse", results['mse'])
                mlflow.log_metric(f"{name}_mae", results['mae'])
                mlflow.log_metric(f"{name}_r2", results['r2'])

        # Save locally as well
        with open("best_model.pkl", "wb") as f:
            pickle.dump(self.best_model, f)

        with open("feature_engineer.pkl", "wb") as f:
            pickle.dump(self.feature_engineer, f)

        logger.info(f"Model saved successfully")

class RiskAssessmentEngine:
    """Main risk assessment engine"""

    def __init__(self):
        self.model = None
        self.feature_engineer = None
        self.explainer = None
        self.redis_client = None
        self._initialize_components()

    def _initialize_components(self):
        """Initialize all components"""
        try:
            # Initialize Redis for caching
            self.redis_client = Redis.from_url(settings.redis_url)

            # Load or train model
            self._load_or_train_model()

            logger.info("Risk assessment engine initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize risk assessment engine: {e}")
            raise

    def _load_or_train_model(self):
        """Load existing model or train new one"""
        try:
            # Try to load existing model
            with open("best_model.pkl", "rb") as f:
                self.model = pickle.load(f)

            with open("feature_engineer.pkl", "rb") as f:
                self.feature_engineer = pickle.load(f)

            logger.info("Loaded existing model")

        except FileNotFoundError:
            logger.info("No existing model found, training new model...")

            # Generate synthetic data and train model
            trainer = ModelTrainer()
            synthetic_data = trainer.generate_synthetic_data(10000)
            trainer.train_models(synthetic_data)
            trainer.save_model()

            self.model = trainer.best_model
            self.feature_engineer = trainer.feature_engineer
            self.explainer = trainer.explainer

            logger.info("New model trained and saved")

    def assess_risk(self, applicant_data: Dict[str, Any]) -> RiskAssessment:
        """Assess risk for an applicant"""
        try:
            # Generate unique applicant ID
            applicant_id = f"app_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{hash(str(applicant_data)) % 10000}"

            # Check cache first
            cached_result = self._get_cached_assessment(applicant_id)
            if cached_result:
                return cached_result

            # Convert to DataFrame
            df = pd.DataFrame([applicant_data])

            # Feature engineering
            df_processed = self.feature_engineer.transform(df)

            # Make prediction
            risk_score = float(self.model.predict(df_processed)[0])

            # Calculate confidence (simplified)
            confidence_score = self._calculate_confidence(df_processed)

            # Determine risk category
            risk_category = self._get_risk_category(risk_score)

            # Get feature importance and explanation
            feature_importance, explanation = self._get_explanation(df_processed, risk_score)

            # Create assessment result
            assessment = RiskAssessment(
                applicant_id=applicant_id,
                risk_score=risk_score,
                risk_category=risk_category,
                confidence_score=confidence_score,
                feature_importance=feature_importance,
                explanation=explanation,
                assessment_timestamp=datetime.now(),
                model_version="v1.0"
            )

            # Cache result
            self._cache_assessment(applicant_id, assessment)

            # Update metrics
            risk_assessment_counter.inc()

            return assessment

        except Exception as e:
            logger.error(f"Risk assessment failed: {e}")
            raise HTTPException(status_code=500, detail=f"Risk assessment failed: {str(e)}")

    def _calculate_confidence(self, df_processed: pd.DataFrame) -> float:
        """Calculate confidence score for the prediction"""
        # Simplified confidence calculation
        # In practice, you might use prediction intervals, ensemble variance, etc.
        return 0.85  # Placeholder

    def _get_risk_category(self, risk_score: float) -> str:
        """Convert risk score to categorical risk level"""
        if risk_score < 20:
            return "very_low"
        elif risk_score < 40:
            return "low"
        elif risk_score < 60:
            return "medium"
        elif risk_score < 80:
            return "high"
        else:
            return "very_high"

    def _get_explanation(self, df_processed: pd.DataFrame, risk_score: float) -> Tuple[Dict[str, float], str]:
        """Generate explanation for the risk assessment"""
        try:
            # Get feature importance using SHAP (if available)
            if self.explainer:
                shap_values = self.explainer(df_processed)
                feature_importance = dict(zip(df_processed.columns, shap_values.values[0]))
            else:
                # Fallback to model feature importance
                if hasattr(self.model, 'feature_importances_'):
                    feature_importance = dict(zip(df_processed.columns, self.model.feature_importances_))
                else:
                    feature_importance = {}

            # Generate human-readable explanation
            explanation = self._generate_explanation_text(feature_importance, risk_score)

            return feature_importance, explanation

        except Exception as e:
            logger.warning(f"Failed to generate explanation: {e}")
            return {}, f"Risk score: {risk_score:.1f} - Explanation unavailable"

    def _generate_explanation_text(self, feature_importance: Dict[str, float], risk_score: float) -> str:
        """Generate human-readable explanation"""
        # Sort features by importance
        sorted_features = sorted(feature_importance.items(), key=lambda x: abs(x[1]), reverse=True)

        explanation_parts = [f"Your risk score is {risk_score:.1f}."]

        if sorted_features:
            explanation_parts.append("Key factors affecting your score:")

            for feature, importance in sorted_features[:5]:  # Top 5 features
                if abs(importance) > 0.01:  # Only significant features
                    direction = "increases" if importance > 0 else "decreases"
                    explanation_parts.append(f"• {feature.replace('_', ' ').title()} {direction} your risk")

        return " ".join(explanation_parts)

    def _get_cached_assessment(self, applicant_id: str) -> Optional[RiskAssessment]:
        """Get cached assessment result"""
        try:
            if self.redis_client:
                cached_data = self.redis_client.get(f"assessment:{applicant_id}")
                if cached_data:
                    data = json.loads(cached_data)
                    return RiskAssessment(**data)
        except Exception as e:
            logger.warning(f"Failed to retrieve cached assessment: {e}")
        return None

    def _cache_assessment(self, applicant_id: str, assessment: RiskAssessment):
        """Cache assessment result"""
        try:
            if self.redis_client:
                # Convert to dict for JSON serialization
                data = asdict(assessment)
                data['assessment_timestamp'] = data['assessment_timestamp'].isoformat()

                # Cache for 1 hour
                self.redis_client.setex(
                    f"assessment:{applicant_id}",
                    3600,
                    json.dumps(data)
                )
        except Exception as e:
            logger.warning(f"Failed to cache assessment: {e}")

# FastAPI Application
app = FastAPI(title="Insurance Risk Assessment API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global engine instance
risk_engine = RiskAssessmentEngine()

@app.post("/assess_risk")
async def assess_risk(request: ApplicantRequest):
    """Assess risk for an insurance applicant"""
    try:
        with risk_assessment_duration.time():
            assessment = risk_engine.assess_risk(request.applicant_data)

        return {
            "applicant_id": assessment.applicant_id,
            "risk_score": assessment.risk_score,
            "risk_category": assessment.risk_category,
            "confidence_score": assessment.confidence_score,
            "feature_importance": assessment.feature_importance if request.explanation_required else None,
            "explanation": assessment.explanation if request.explanation_required else None,
            "assessment_timestamp": assessment.assessment_timestamp.isoformat(),
            "model_version": assessment.model_version
        }

    except Exception as e:
        logger.error(f"Risk assessment API error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "timestamp": datetime.now().isoformat()}

@app.get("/model_info")
async def get_model_info():
    """Get information about the current model"""
    return {
        "model_type": type(risk_engine.model).__name__,
        "model_version": "v1.0",
        "features_count": len(risk_engine.feature_engineer.scalers.get('numeric', [])),
        "last_updated": datetime.now().isoformat()
    }

# Model retraining endpoint (for batch updates)
@app.post("/retrain_model")
async def retrain_model(training_data: List[Dict[str, Any]]):
    """Retrain the model with new data"""
    try:
        # Convert to DataFrame
        df = pd.DataFrame(training_data)

        # Retrain model
        trainer = ModelTrainer()
        trainer.train_models(df)
        trainer.save_model()

        # Update engine with new model
        risk_engine.model = trainer.best_model
        risk_engine.feature_engineer = trainer.feature_engineer
        risk_engine.explainer = trainer.explainer

        return {"message": "Model retrained successfully", "timestamp": datetime.now().isoformat()}

    except Exception as e:
        logger.error(f"Model retraining failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# Batch assessment endpoint
@app.post("/assess_risk_batch")
async def assess_risk_batch(requests: List[Dict[str, Any]]):
    """Assess risk for multiple applicants"""
    try:
        results = []

        for applicant_data in requests:
            assessment = risk_engine.assess_risk(applicant_data)
            results.append({
                "applicant_id": assessment.applicant_id,
                "risk_score": assessment.risk_score,
                "risk_category": assessment.risk_category,
                "confidence_score": assessment.confidence_score
            })

        return {"assessments": results, "total_processed": len(results)}

    except Exception as e:
        logger.error(f"Batch risk assessment failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    # Initialize database and other components
    logger.info("Starting Insurance Risk Assessment Engine...")

    # Run the FastAPI application
    uvicorn.run(app, host="0.0.0.0", port=8000)

2025-07-08 08:19:45 [info     ] No existing model found, training new model...
2025-07-08 08:19:45 [info     ] Starting model training...
2025-07-08 08:19:46 [info     ] Training linear_regression...
2025-07-08 08:19:47 [info     ] linear_regression - MSE: 6.9431, MAE: 2.0862, R2: 0.8840
2025-07-08 08:19:47 [info     ] Training random_forest...
2025-07-08 08:22:03 [info     ] random_forest - MSE: 5.0229, MAE: 1.7536, R2: 0.9161
2025-07-08 08:22:03 [info     ] Training gradient_boosting...
2025-07-08 08:22:45 [info     ] gradient_boosting - MSE: 4.2087, MAE: 1.6063, R2: 0.9297
2025-07-08 08:22:45 [info     ] Training xgboost...
2025-07-08 08:22:53 [info     ] xgboost - MSE: 5.0376, MAE: 1.7631, R2: 0.9158
2025-07-08 08:22:53 [info     ] Training lightgbm...
2025-07-08 08:22:56 [info     ] lightgbm - MSE: 4.4890, MAE: 1.6632, R2: 0.9250
2025-07-08 08:22:56 [info     ] Best model: gradient_boosting
2025-07-08 08:22:56 [error    ] Failed to initialize risk assessment engine: Cannot cast ar

TypeError: Cannot cast array data from dtype('O') to dtype('float64') according to the rule 'safe'